In [10]:
import pandas as pd
import numpy as np

# Load the raw dataset from Day 1
df = pd.read_csv("../data/raw/cardekho_raw.csv")
print(f"Starting shape: {df.shape[0]} rows, {df.shape[1]} columns")

Starting shape: 8128 rows, 13 columns


In [11]:
# Count and remove exact duplicate rows
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Removed {duplicates_count} duplicate rows.")
print(f"New shape: {df.shape[0]} rows, {df.shape[1]} columns")

Removed 1202 duplicate rows.
New shape: 6926 rows, 13 columns


In [12]:
# Drop 'torque' because its units and RPM ranges are unstructured and redundant with engine & max_power
df = df.drop(columns=['torque'])
print("Remaining columns:", list(df.columns))

Remaining columns: ['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'seats']


In [13]:
# Strip units ('kmpl', 'km/kg', 'CC', 'bhp') by splitting at the space and keeping the first part
df['mileage'] = pd.to_numeric(df['mileage'].str.split().str[0], errors='coerce')
df['engine'] = pd.to_numeric(df['engine'].str.split().str[0], errors='coerce')
df['max_power'] = pd.to_numeric(df['max_power'].str.split().str[0], errors='coerce')

# Verify they are now float64 numbers!
print(df[['mileage', 'engine', 'max_power']].dtypes)
df[['mileage', 'engine', 'max_power']].head()

mileage      float64
engine       float64
max_power    float64
dtype: object


,mileage,engine,max_power
0,23.40,1248.0,74.00
1,21.14,1498.0,103.52
2,17.70,1497.0,78.00
3,23.00,1396.0,90.00
4,16.10,1298.0,88.20


In [14]:
# Strip any accidental leading/trailing spaces in text columns
cat_cols = ['name', 'fuel', 'seller_type', 'transmission', 'owner']
for col in cat_cols:
    df[col] = df[col].str.strip()

print("Owner counts BEFORE cleaning:")
print(df['owner'].value_counts())

# Remove 'Test Drive Car' rows since our app predicts USED car resale prices
df = df[df['owner'] != 'Test Drive Car'].reset_index(drop=True)

print("\nOwner counts AFTER removing Test Drive Cars:")
print(df['owner'].value_counts())

Owner counts BEFORE cleaning:
owner
First Owner             4242
Second Owner            1974
Third Owner              536
Fourth & Above Owner     169
Test Drive Car             5
Name: count, dtype: int64

Owner counts AFTER removing Test Drive Cars:
owner
First Owner             4242
Second Owner            1974
Third Owner              536
Fourth & Above Owner     169
Name: count, dtype: int64


In [15]:
# Check summary statistics of all numerical columns
print(df.describe().round(2))

# Check how many cars have an impossible 0 kmpl mileage
zero_mileage_count = (df['mileage'] == 0).sum()
print(f"\nCars with 0.0 mileage (data entry error): {zero_mileage_count}")

# Replace 0.0 mileage with NaN so the ML pipeline imputes a realistic median later
df.loc[df['mileage'] == 0, 'mileage'] = np.nan

# Also check and replace impossible 0 bhp max_power with NaN
print(f"Cars with 0.0 max_power (data entry error): {(df['max_power'] == 0).sum()}")
df.loc[df['max_power'] == 0, 'max_power'] = np.nan

          year  selling_price   km_driven  mileage   engine  max_power  \
count  6921.00        6921.00     6921.00  6713.00  6713.00    6715.00   
mean   2013.42      514462.90    74038.56    19.47  1430.66      87.67   
std       4.08      505754.41    58356.94     4.05   493.59      31.71   
min    1983.00       29999.00        1.00     0.00   624.00       0.00   
25%    2011.00      250000.00    40000.00    16.80  1197.00      67.10   
50%    2014.00      400000.00    70000.00    19.44  1248.00      81.83   
75%    2017.00      630000.00   100000.00    22.50  1498.00     100.00   
max    2020.00    10000000.00  2360457.00    42.00  3604.00     400.00   

         seats  
count  6713.00  
mean      5.43  
std       0.98  
min       2.00  
25%       5.00  
50%       5.00  
75%       5.00  
max      14.00  

Cars with 0.0 mileage (data entry error): 15
Cars with 0.0 max_power (data entry error): 3


In [16]:
# Verify missing values (these will be imputed inside the Scikit-learn pipeline on Day 4 AFTER train-test split)
print("Missing values left for the Day 4 Pipeline (to prevent data leakage):")
print(df.isnull().sum())

# Save the cleaned dataset
OUTPUT_PATH = "../data/processed/cardekho_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUTPUT_PATH} with shape: {df.shape}")

Missing values left for the Day 4 Pipeline (to prevent data leakage):
name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          223
engine           208
max_power        209
seats            208
dtype: int64

Saved cleaned dataset to ../data/processed/cardekho_cleaned.csv with shape: (6921, 12)
